# 02 -- Baseline CNN: a from-scratch sanity check

Before reaching for transfer learning or transformers, train the
simplest reasonable thing: a 4-block convolutional network with no
pretraining, on log-mel spectrograms. This establishes the floor every
fancier model in this project has to beat -- if a pretrained ResNet or an
84M-parameter transformer can't clear this baseline by a real margin, the
extra complexity isn't earning its keep on a 54-species, severely
imbalanced problem.

The model lives in `watkins/models/cnn_baseline.py`
(`BaselineCNN`: conv-bn-relu-maxpool x4, global average pool, linear
head, ~403K parameters for 54 classes).

In [ ]:
import os
import subprocess
import sys
import importlib.util

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Mount Drive and locate the project. Upload src/, configs/,
    #    pyproject.toml, requirements.txt to this path in My Drive first --
    #    NOT the multi-GB Watkins/ or results/ folders, those are handled
    #    separately below.
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DRIVE_PATH = "/content/drive/MyDrive/ShipsEAR"  # <-- edit if you used a different path
    if not os.path.exists(f"{PROJECT_DRIVE_PATH}/src/watkins"):
        raise FileNotFoundError(
            f"Expected the project's src/ folder at {PROJECT_DRIVE_PATH}/src on Google Drive.\n"
            "Upload src/, configs/, pyproject.toml, and requirements.txt there "
            "(skip the multi-GB Watkins/ and results/ folders), or edit "
            "PROJECT_DRIVE_PATH above to match where you put them."
        )
    sys.path.insert(0, f"{PROJECT_DRIVE_PATH}/src")

    # 2. Install whatever Colab's base image doesn't already have. Deliberately
    #    does NOT touch torch/torchaudio/torchvision -- Colab's preinstalled
    #    versions are already matched to its GPU + CUDA build, and reinstalling
    #    this project's CPU-only wheels here would silently disable the GPU.
    needed = ["transformers", "timm", "soundfile", "datasets", "huggingface_hub", "pyarrow"]
    missing = [pkg for pkg in needed if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

    # 3. Data goes on fast local/ephemeral disk (re-derivable from the public
    #    Hugging Face source, no reason to pay Drive's slow random-I/O tax on
    #    thousands of per-epoch file reads); results go on Drive so trained
    #    checkpoints/metrics survive a runtime disconnect.
    os.environ["WATKINS_DATA_ROOT"] = "/content/watkins_data"
    os.environ["WATKINS_RESULTS_ROOT"] = f"{PROJECT_DRIVE_PATH}/results"

    from watkins.data import DATA_ROOT
    if not (DATA_ROOT / "manifest.csv").exists():
        print("Materializing the Watkins dataset locally -- one-time per Colab runtime, ~10-15 min...")
        subprocess.run([sys.executable, "-m", "watkins.prepare_data"], check=True)
else:
    sys.path.insert(0, "../src")

from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

from watkins.train import train_run, load_config


## 1. Train it

This project's training loop (`watkins.train.train_run`) is the same
code the CLI (`python -m watkins.train --config ...`) uses -- calling it
directly from the notebook just makes results immediately inspectable.

The full config (`configs/baseline_cnn.yaml`) trains up to 40 epochs
with early stopping on the tape-grouped split (see notebook 00 for why
that grouping, rather than a naive per-clip split, is the honest choice).
That can take a while on CPU across ~15,000 clips. For an interactive
first pass, run a short demo here with a reduced epoch count and a data
subset; run the full config from a terminal (`python -m watkins.train
--config configs/baseline_cnn.yaml`) when you want the real number,
ideally in the background while you keep working through later
notebooks.

In [ ]:
demo_cfg = load_config("../configs/baseline_cnn.yaml")
demo_cfg["run_name"] = "baseline_cnn_demo"
demo_cfg["epochs"] = 5
demo_cfg["subset_frac"] = 0.3  # faster demo; drop this for the real run

demo_result = train_run(demo_cfg)
print("demo test accuracy:", demo_result["test_acc"], " macro F1:", demo_result["test_f1"])


## 2. Look at the learning curves

A model that's still improving on validation loss when training stops is
under-trained; a model whose val loss is climbing while train loss falls
is overfitting. With ~9,000-10,000 training clips (tape-grouped split)
and a 403K-parameter model, some overfitting is expected -- that's what
`class_weighted_loss`, `noise_augment_p`, and `spec_augment` in the config
are there to fight.

In [ ]:
log = pd.read_csv(demo_result["log_path"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(log["epoch"], log["train_loss"], label="train")
axes[0].plot(log["epoch"], log["val_loss"], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("loss")
axes[1].plot(log["epoch"], log["train_f1"], label="train macro-F1")
axes[1].plot(log["epoch"], log["val_f1"], label="val macro-F1")
axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].set_title("macro F1")
plt.tight_layout()
plt.show()


## 3. Per-class metrics and confusion matrix

This is where accuracy alone would mislead you. `watkins.evaluate`
reports precision/recall/F1 per class and a confusion matrix -- look for
which species get confused with which. A common pattern worth checking:
does the model favor killer whale or sperm whale (the two largest
species by clip count) at the expense of the rarer ones?

In [ ]:
from watkins.evaluate import evaluate_checkpoint

eval_result = evaluate_checkpoint(demo_result["checkpoint_path"])


The confusion matrix figure was also saved to
`results/figures/baseline_cnn_demo_confusion.png`.

## 4. The leakage gap, made concrete

Notebook 00 argued that a naive per-clip split lets a model partly
recognize a *recording* instead of a *species*, because clips from the
same tape end up in both train and test. Let's quantify what that's worth
in accuracy points, using this same baseline model: train once on
`tape_grouped` (already done above) and once on the deliberately leaky
`clip_random` mode, then compare test accuracy.

In [ ]:
leak_cfg = load_config("../configs/baseline_cnn.yaml")
leak_cfg["run_name"] = "baseline_cnn_leak_check"
leak_cfg["epochs"] = 5
leak_cfg["subset_frac"] = 0.3
leak_cfg["split_mode"] = "clip_random"

leak_result = train_run(leak_cfg)
print(f"tape_grouped split test acc: {demo_result['test_acc']:.3f}")
print(f"clip_random (leaky) split test acc: {leak_result['test_acc']:.3f}")
print(f"gap: {leak_result['test_acc'] - demo_result['test_acc']:+.3f}")


If the `clip_random`-split number comes out noticeably higher, that gap
*is* the leakage -- the model isn't smarter under that split, it's being
tested on near-duplicates of its own training data (same tape, same
individual animal, same recording noise floor). Keep this firmly in mind
whenever you see a bioacoustic classification accuracy number without
knowing how the underlying data was split.

## Exercises

1. Re-run the demo with `class_weighted_loss: False`. Does overall
   accuracy go up while macro-F1 goes down? That's the imbalance trap in
   action -- explain in your own words why that combination can happen
   on a dataset where killer whale alone is ~28% of the training set.
2. Turn off `spec_augment` and `noise_augment_p` (set both to
   inert values) and compare val-loss curves. Does the model overfit
   faster without them?
3. Run the full (non-demo) config to completion
   (`python -m watkins.train --config configs/baseline_cnn.yaml`, or
   drop `subset_frac`/lower `epochs` overrides here) and re-evaluate.
   How much does the extra data and training time move macro-F1 versus
   the 5-epoch/30%-data demo?
4. Look at the confusion matrix: which species pair is most confused?
   Given what you know from `docs/class_reference.md` about which
   species have few tapes (little acoustic variety to learn from) versus
   many, does the confusion pattern correlate with tape count rather than
   just clip count?